In [ ]:
# ===================================================================================
# DASHBOARD MIGRATION SCRIPT V13.1 (STRICT SHARING + GROUPS + FIXES)
# - FIX: Corrected Folder object access logic.
# - FIX: Solves "SharingGroupManager" and dictionary parsing errors for groups.
# - UPDATE: Strict Sharing Mirroring (Private stays Private).
# - UPDATE: Mirrors Group Sharing (Matches Target Groups by Title).
# ===================================================================================

import pandas as pd
import json
import copy
import time
import csv
import os
import datetime
import urllib3
import requests
import sys
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC ---
# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_3_dashboards.json")
if os.path.exists(_sidecar_path):
    import json as _json
    DASHBOARD_IDS_TO_MIGRATE = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(DASHBOARD_IDS_TO_MIGRATE)} Dashboard IDs from sidecar.")
else:
    DASHBOARD_IDS_TO_MIGRATE = [
        # Example Source IDs
        # "845de10d90e04a6d81c2c7bd4b1860c1", 
        # "a0b94b2358ae466cb806208412cb343f",
    ]
    
# =============================================================================
# --- CONNECT (HARDENED) -------------------------------------------------------
# =============================================================================
# Initialize variables to None to prevent NameError
source_gis = None
target_gis = None

print("Connecting...")
try:
    session = requests.Session()
    # Robust retry strategy for 500 errors
    retry_strategy = Retry(
        total=5, 
        backoff_factor=2, 
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "POST"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)
    
    # Verify connections explicitly
    print(f"   > Source Connected: {source_gis.url}")
    if not target_gis.users.me: raise Exception("Target login failed.")
    print(f"   > Target Connected: {target_gis.users.me.username}")

except Exception as e:
    print(f"\n❌ CRITICAL CONNECTION FAILURE: {e}")
    print("   The script cannot continue because the GIS object failed to initialize.")
    sys.exit(1) # Force exit

# Double check safety before loop
if not source_gis or not target_gis:
    print("❌ GIS Initialization incomplete. Aborting.")
    sys.exit(1)

# =============================================================================
# --- LOGGING & HISTORY --------------------------------------------------------
# =============================================================================
STATS = {"scanned": 0, "created": 0, "failed": 0, "skipped_log": 0}
ALREADY_MIGRATED_IDS = set()

if os.path.exists(LOG_FILE):
    try:
        df = pd.read_csv(LOG_FILE)
        if 'SourceID' in df.columns: ALREADY_MIGRATED_IDS = set(df['SourceID'].astype(str).str.strip())
    except: pass
else:
    # Create log file if it doesn't exist
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except: pass

def log_migration(source_id, target_id, title):
    try:
        with open(LOG_FILE, 'a', newline='') as f:
            csv.writer(f).writerow([source_id, "N/A", target_id, title, datetime.datetime.now(), "Dashboard"])
            ALREADY_MIGRATED_IDS.add(str(source_id))
    except: pass

# =============================================================================
# --- HELPER: FOLDER & OWNER UTILS (FIXED) -------------------------------------
# =============================================================================
def get_source_folder_name(source_item):
    """Retrieve the folder name of the source item."""
    try:
        if not source_item.ownerFolder: return None
        user = source_gis.users.get(source_item.owner)
        if user:
            for f in user.folders:
                # Handle both dict and object access
                fid = f['id'] if isinstance(f, dict) else f.id
                ftitle = f['title'] if isinstance(f, dict) else f.title
                if fid == source_item.ownerFolder: return ftitle
    except: pass
    return None

def ensure_target_folder_exists(username, folder_name):
    """Ensure a folder exists for a specific user in Target."""
    try:
        user = target_gis.users.get(username)
        if not user: return False
        
        # Check existing folders (Handles Folder Objects AND Dicts)
        existing_folders = []
        for f in user.folders:
            if hasattr(f, 'title'):
                existing_folders.append(f.title)
            elif isinstance(f, dict):
                existing_folders.append(f.get('title'))
        
        if folder_name in existing_folders: return True
        
        # Create if missing
        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except: return False

def assign_correct_owner_and_folder(source_item, target_item):
    """
    Tries to match Source Owner in Target.
    If found -> Moves to same folder name.
    If NOT found -> Moves to DEFAULT_OWNER (portaladm) and DEFAULT_FOLDER (migrate_test).
    """
    try:
        source_owner = source_item.owner
        target_owner_to_use = DEFAULT_OWNER
        target_folder_to_use = DEFAULT_FOLDER

        # Check if Source Owner exists in Target
        found_user = target_gis.users.get(source_owner)
        
        if found_user:
            print(f"      👤 Source owner '{source_owner}' found in target.")
            target_owner_to_use = source_owner
            
            # Try to get source folder name
            src_folder_name = get_source_folder_name(source_item)
            if src_folder_name:
                if ensure_target_folder_exists(target_owner_to_use, src_folder_name):
                    target_folder_to_use = src_folder_name
                else:
                    print(f"      ⚠ Could not create folder '{src_folder_name}'. Using Default.")
            else:
                target_folder_to_use = None # Root
        else:
            print(f"      👤 Source owner '{source_owner}' NOT found. Defaulting to '{DEFAULT_OWNER}'.")
            # Ensure default folder exists for default user
            ensure_target_folder_exists(DEFAULT_OWNER, DEFAULT_FOLDER)

        # Reassign
        print(f"      📂 Moving to: Owner={target_owner_to_use}, Folder={target_folder_to_use}")
        target_item.reassign_to(target_owner_to_use, target_folder=target_folder_to_use)

    except Exception as e:
        print(f"      ⚠ Reassign/Move Failed: {e} (Item remains with Creator)")

# =============================================================================
# --- HELPER: MIRROR SHARING (FIXED DICTIONARY EXTRACTION) ---------------------
# =============================================================================
def mirror_source_sharing(source_item, target_item):
    """
    1. Checks Source Sharing (Public/Org/Private).
    2. Maps Source Groups -> Target Groups (by exact Title).
    3. Handles both List and Dictionary return types from 'shared_with'.
    """
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")
        
        # 1. Access Level
        is_public = False
        is_org = False
        if source_item.access == 'public':
            is_public = True
            is_org = True 
        elif source_item.access == 'org':
            is_org = True
            
        # 2. Get List of Groups (Correctly parse the dictionary)
        source_group_list = []
        raw_shared = source_item.shared_with
        
        if isinstance(raw_shared, dict) and 'groups' in raw_shared:
            # This is the standard behavior: {'everyone': F, 'org': F, 'groups': [List]}
            source_group_list = raw_shared['groups']
        elif isinstance(raw_shared, list):
            # Rare fallback
            source_group_list = raw_shared
            
        target_group_ids = []
        
        if source_group_list:
            print(f"         - Found {len(source_group_list)} source groups. Mapping...")
            for sg in source_group_list:
                # Handle Group Object vs String Name
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)

                # Search for group with exact title in Target
                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                
                # Filter for exact match
                matched_group = next((g for g in found_groups if g.title == sg_title), None)
                
                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")
        
        # 3. Apply
        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")

    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")

# =============================================================================
# --- HELPER: RECURSIVE SWAP WITH GAP PATCH ------------------------------------
# =============================================================================
def find_and_swap_ids(obj):
    changes_made = False
    if isinstance(obj, dict):
        # 1. Swap Service ID
        if 'itemId' in obj:
            old_id = obj['itemId']
            
            # Search Target for the migrated equivalent
            search_query = f'tags:"SourceID:{old_id}"'
            try:
                results = target_gis.content.search(search_query, max_items=1)
                
                if results:
                    print(f"     >> SWAPPING Dependency: {old_id} -> {results[0].id}")
                    obj['itemId'] = results[0].id
                    changes_made = True
                    
                    # --- GAP FIX (disabled) ---
                    # Uncomment and set PROBLEM_SOURCE_ID in migration_config.py if your
                    # source service has a layer-index gap.
                    # if old_id == PROBLEM_SOURCE_ID:
                    #     if 'layerId' in obj:
                    #         try:
                    #             lid = int(obj['layerId'])
                    #             if lid > 17:
                    #                 print(f"     🔧 Patching Dashboard Layer: {lid} -> {lid - 1}")
                    #                 obj['layerId'] = lid - 1
                    #                 changes_made = True
                    #         except: pass
            except Exception as e:
                print(f"     ⚠ Swap Error for {old_id}: {e}")

        # Recursive search for nested dictionaries
        for k, v in obj.items():
            if find_and_swap_ids(v): changes_made = True

    elif isinstance(obj, list):
        # Recursive search for lists
        for item in obj:
            if find_and_swap_ids(item): changes_made = True
            
    return changes_made

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print(f"Starting Dashboard Migration (V13.1)...")
start_time = time.time()

for dash_id in DASHBOARD_IDS_TO_MIGRATE:
    try:
        STATS["scanned"] += 1
        if str(dash_id) in ALREADY_MIGRATED_IDS:
             print(f"\n[Skip] {dash_id} found in log.")
             STATS["skipped_log"] += 1
             continue

        src = source_gis.content.get(dash_id)
        if not src:
            print(f"❌ Source {dash_id} not found.")
            STATS["failed"] += 1
            continue

        print(f"\nProcessing: {src.title}...")
        
        # --- DETERMINE TARGET TITLE ---
        target_title = src.title
        if APPEND_MIGRATED:
            target_title = f"{src.title} (Migrated)"

        # Search for existence using the TARGET title
        existing = target_gis.content.search(f'title:"{target_title}"', max_items=1)
        if existing:
            print(f"   ⚠️ Dashboard already exists in Target. Skipping.")
            STATS["skipped_log"] += 1
            continue

        data = src.get_data()
        if not data: continue

        # Swap IDs
        new_data = copy.deepcopy(data)
        find_and_swap_ids(new_data)

        # --- PREPARE TAGS (Fix: Preserve Original + Add Migration Tags) ---
        final_tags = list(src.tags) if src.tags else []
        if "Migrated" not in final_tags: final_tags.append("Migrated")
        tag_id = f"SourceID:{dash_id}"
        if tag_id not in final_tags: final_tags.append(tag_id)

        props = {
            "type": "Dashboard",
            "title": target_title,
            "snippet": src.snippet,
            "description": src.description,
            "tags": final_tags,
            "text": json.dumps(new_data)
        }

        # Ghost Buster Creation
        new_dash = None
        try:
            # Create in default folder first, move later
            # Use content.add instead of folder.add to avoid deprecation warning
            new_dash = target_gis.content.add(props, folder=DEFAULT_FOLDER)
        except Exception as e:
            if "already exists" in str(e).lower():
                print("   ⚠ Name collision. Retrying with timestamp...")
                props["title"] = f"{target_title} {int(time.time())}"
                new_dash = target_gis.content.add(props, folder=DEFAULT_FOLDER)
            else: raise e

        # Thumbnails & Metadata
        try:
            temp = "thumbs"
            os.makedirs(temp, exist_ok=True)
            p = src.download_thumbnail(save_folder=temp)
            if p: new_dash.update(thumbnail=p)
        except: pass

        # --- SHARE WITH ORG & GROUPS (STRICT MIRROR) ---
        # Share BEFORE reassigning owner to avoid privilege errors
        mirror_source_sharing(src, new_dash)
        
        # Owner & Folder Assignment
        assign_correct_owner_and_folder(src, new_dash)

        log_migration(dash_id, new_dash.id, new_dash.title)
        STATS["created"] += 1
        print(f"🚀 SUCCESS: Created '{new_dash.title}'")
        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ Failed: {e}")
        STATS["failed"] += 1

# =============================================================================
# --- FINAL REPORT -------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "="*60)
print("        DASHBOARD MIGRATION REPORT (V13.1)     ")
print("="*60)
print(f" ⏱️  Total Duration:            {duration_str}")
print("-" * 60)
print(f" 📊 Dashboards Scanned:       {STATS['scanned']}")
print(f" ⏭️  Skipped (CSV):             {STATS['skipped_log']}")
print(f" ✅ Dashboards Created:       {STATS['created']}")
print(f" ❌ Failures:                 {STATS['failed']}")
print("="*60)